# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: content refresh** — deciding which existing content pieces on a client's site are losing traffic and should be prioritized for a rewrite/update.

**Task type: scoring (a ranking problem expressed as a continuous score).**

Not classification, because a yes/no "needs refresh" flag throws away *how bad* the decline is, and refresh work is done under a limited budget (a team can only rewrite so many pages a month) — so what actually matters is a rank-ordered priority list, not a label. Not clustering, because I already have a clear notion of "good" (traffic holding/growing) vs "bad" (traffic collapsing) — I'm not discovering unlabeled groups. I'm treating it as a **scoring** task rather than pure ranking because I want a number I can also threshold, log, and explain to a client ("this page scored in the bottom 10%"), not just a relative order.

Evidence this is a real spread, not a coin flip: `trend_direction` (down/stable/up/new/flat) already shows most content trending in one of five buckets below, and `trend_pct` (the underlying % change in sessions vs. the prior 30 days) is a continuous number with real variance — exactly what a scoring model would predict.

**Action the output supports:** the score becomes a sorted queue. Each week, the content team pulls the top N highest-priority (worst-scoring) pages off the queue and sends them out for a rewrite. It's a decision-support ranking, not an automated rewrite trigger.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(df["trend_direction"].value_counts())
print()
print(df["trend_pct"].describe())


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy): `trend_pct`** — the percent change in `sessions_last_30d` vs `sessions_prev_30d`, i.e. how much traffic to that piece of content grew or shrank month-over-month.

This is a **proxy, not a direct label**. The thing I actually care about — "is this content stale/wrong/underperforming and worth a rewrite" — isn't directly observed anywhere in the data. Nobody hand-labeled "needs refresh." Instead I'm using an **observed outcome** (traffic decline) that correlates with content going stale, but it isn't the same thing:
- A page can decline because a competitor outranked it (not because our content is bad)
- A page can decline just because seasonal search volume dropped
- A page can stay flat while still containing outdated information nobody is searching for yet

So the honest framing is: I'm predicting a *traffic-decline proxy* for refresh priority, and I should say "content whose traffic is declining" rather than claim I'm detecting "outdated content" directly.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

target = df["trend_pct"]
print("rows with a usable target:", target.notna().sum(), "/", len(df))
print(target.describe())
print()
print("median session change: content typically lost", 
      f"{-target.median():.1f}% of its sessions month-over-month")


rows with a usable target: 26612 / 30000
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

median session change: content typically lost 33.5% of its sessions month-over-month


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: top-decile capture rate** — of the 10% of content that actually declined the most (worst `trend_pct`), what fraction does my model correctly place in its own worst-scoring 10%?

I'm not using raw error (MAE/RMSE) on `trend_pct` as the headline metric, because `trend_pct` has extreme outliers (some values run into the tens of thousands of percent from tiny denominators), which would let a model "win" on average error while being useless for the actual decision. The actual decision is a budget-constrained priority list — "which 10% of pages do we rewrite this month" — so I want a metric that directly measures whether the model's ranking puts the truly worst content at the top.

'Good' = a capture rate meaningfully above what a naive rule achieves (see Section 5 for that baseline number).

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
sub = df.dropna(subset=["trend_pct"]).copy()

worst_cutoff = sub["trend_pct"].quantile(0.10)
sub["is_worst_decile"] = sub["trend_pct"] <= worst_cutoff

print(f"worst-decile threshold: trend_pct <= {worst_cutoff:.1f}%")
print(f"content in the worst decile: {sub['is_worst_decile'].sum()} of {len(sub)} rows")
print()
print("A model scores 'good' here if, among the content it ranks in its own worst 10%,")
print("a large majority actually falls in this true worst decile.")


worst-decile threshold: trend_pct <= -85.4%
content in the worst decile: 2662 of 26612 rows

A model scores 'good' here if, among the content it ranks in its own worst 10%,
a large majority actually falls in this true worst decile.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one piece of published content** (`content_id`) belonging to one client (`client_id`), described by its SEO context (search volume, competition, intent), how it was produced (`provider_used`, `model_used`), its size, how old/fresh it is, and 90-day traffic + engagement metrics.

In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

lane_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "content_age_days", "days_since_last_update", "freshness_tier",
    "avg_position", "ctr", "engagement_rate", "ai_traffic_pct",
    "sessions_last_30d", "sessions_prev_30d", "trend_direction", "trend_pct",
]

lane = df[lane_cols]
print("shape:", lane.shape)
lane.head()


shape: (30000, 16)


,content_id,client_id,content_type,main_intent,word_count,content_age_days,days_since_last_update,freshness_tier,avg_position,ctr,engagement_rate,ai_traffic_pct,sessions_last_30d,sessions_prev_30d,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,20,0-30,10.6,0.76,5.88,0.0,2,9,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,25,0-30,20.3,0.05,0.00,0.0,3,2,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,20,0-30,36.5,0.09,0.00,0.0,1,3,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,22,0-30,6.2,0.49,1.28,0.0,35,26,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,14,0-30,44.0,0.13,0.00,0.0,14,9,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


The obvious fixed rule is: *"flag content as needing a refresh if it hasn't been updated in a while (e.g. `days_since_last_update > 180`)."* I tested that rule against what actually happened to traffic, and it fails badly:

- Correlation between `days_since_last_update` and `trend_pct` is essentially **zero** — staleness by the calendar doesn't predict the traffic outcome
- The same near-zero pattern holds for `avg_position`, `ctr`, `ai_traffic_pct`, and `content_age_days` **on their own**
- The stale-content rule only catches a small fraction of the pages that are actually in the worst-declining decile

No single column carries the signal alone. What predicts decline is *interactions* — e.g. old content that's also low word count and losing rank position, or content with high AI-answer traffic share (an audience is reading a summary and never clicking through) combined with weak engagement. That combination is exactly the kind of multi-variable, non-linear pattern an if/else rule can't express but a model that weighs and combines features can.

In [5]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
sub = df.dropna(subset=["trend_pct"]).copy()

for col in ["days_since_last_update", "avg_position", "ctr", "ai_traffic_pct", "content_age_days"]:
    print(f"corr({col}, trend_pct) = {sub[col].corr(sub['trend_pct']):.4f}")

print()
worst_cutoff = sub["trend_pct"].quantile(0.10)
worst = sub["trend_pct"] <= worst_cutoff
rule_flag = sub["days_since_last_update"] > 180

caught = (worst & rule_flag).sum()
print(f"'stale content' rule (days_since_last_update > 180) catches "
      f"{caught} of {worst.sum()} truly worst-declining pages "
      f"({caught / worst.sum():.1%})")


corr(days_since_last_update, trend_pct) = -0.0142
corr(avg_position, trend_pct) = 0.0472
corr(ctr, trend_pct) = 0.0080
corr(ai_traffic_pct, trend_pct) = -0.0038
corr(content_age_days, trend_pct) = 0.0007

'stale content' rule (days_since_last_update > 180) catches 25 of 2662 truly worst-declining pages (0.9%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.